In [ ]:
import sys
import os

# go two levels up to reach the project root

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if project_root not in sys.path:
    sys.path.append(project_root)

from modules.utils.date_utils import get_month_start_n_months_ago
from pyspark.sql.functions import date_format

In [ ]:
# get first day of the month 4 months ago

four_month_ago_start = get_month_start_n_months_ago(4)

print(four_month_ago_start)

In [ ]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()

df = spark.read.table("nyctaxi.02_silver.yellow_trips_enriched").filter(f"tpep_pickup_datetime > '{four_month_ago_start}'")

df.show()


In [ ]:
# add year-month column

df = df.withColumn("year_month", date_format("tpep_pickup_datetime", "yyyy-MM"))

In [ ]:
# write dataframe into the external table in json format

df.write.\
    option("path", "abfss://nyctaxi-yellow@nyctaxistorage0000001.dfs.core.windows.net/yellow_trips_export/").\
    format("json").\
    mode("append").\
    partitionBy("vendor","year_month").\
    saveAsTable("nyctaxi.04_export.yellow_trips_export")